# From CIFAR-10 / ResNet-20 to MNIST / LeNet-5 with ***torchdistill***

torchdistill ships an example config that trains ResNet-20 on CIFAR-10. In this notebook we reuse it to train a
**different model (LeNet-5) on a different dataset (MNIST)** without editing a single line of torchdistill's code.
We only add

1. **a model file**, `lenet5.py`, placed next to the example script and registered with `@register_model`, and
2. **a config**, `lenet5_mnist-hyperparameter_tuning.yaml`, made by editing the CIFAR-10 / ResNet-20 config.

The training script, `examples/torchvision/image_classification.py`, stays exactly as it is.

**Runtime:** a CPU runtime is enough (*Runtime → Change runtime type → CPU*). The whole notebook takes a few minutes.

## 1. Install torchdistill and get its example code

We pin torchdistill **v1.1.5** so that the pip package and the cloned example script and configs match.

In [ ]:
%pip install -q torchdistill==1.1.5

In [ ]:
!git clone -q --depth 1 --branch v1.1.5 https://github.com/yoshitomo-matsubara/torchdistill.git

If pip reports that it upgraded `torch`, restart the session (*Runtime → Restart session*) before you continue.

In [ ]:
import difflib
import importlib.util
import re
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import torchdistill
from torchdistill.common.main_util import load_ckpt
from torchdistill.common.yaml_util import load_yaml_file
from torchdistill.models.registry import get_model

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'torchdistill {torchdistill.__version__}, torch {torch.__version__}, device: {DEVICE}')

## 2. The model: one new file next to the example script

When Python runs a script, it puts the script's directory on `sys.path`. Any module saved next to
`image_classification.py` can therefore be imported by name, and the config's `dependencies` entry (step 3) does that.
`@register_model` then adds `LeNet5` to torchdistill's model registry under its class name, so the config can ask
for it with `key: 'LeNet5'`.

LeNet-5 was designed for 32×32 inputs. `padding=2` in the first convolution lets it take MNIST's 28×28 digits as they
are.

In [ ]:
%%writefile torchdistill/examples/torchvision/lenet5.py
from torch import nn
from torchdistill.models.registry import register_model


@register_model
class LeNet5(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        # padding=2 makes 28x28 inputs match the original's 32x32
        self.features = nn.Sequential(
            nn.Conv2d(1, 6, 5, padding=2), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(6, 16, 5), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(16 * 5 * 5, 120), nn.ReLU(inplace=True),
            nn.Linear(120, 84), nn.ReLU(inplace=True), nn.Linear(84, num_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))

### What does `%%writefile` do?

The cell above starts with `%%writefile torchdistill/examples/torchvision/lenet5.py`. Lines that start with `%` or `%%`
are not Python: they are **magic commands**, shortcuts that Jupyter and Colab understand (you already used one,
`%pip`). `%%writefile <path>` must be the very first line of a cell, and it changes what running the cell does:

- the code below that first line is **not executed**. It is **saved as text** to the file `<path>`,
- the file is created if it does not exist (Colab prints `Writing ...`) and replaced if it does (`Overwriting ...`).

Why not just run the model code here? The training in step 4 runs `image_classification.py` as a **separate
program**, and that program can only use code that is saved in a file. A class defined only in this notebook would be
invisible to it.

To change the model later, edit the cell and run it again: that rewrites the file. You can open the files it creates
from the **Files** panel (the folder icon on Colab's left sidebar).

## 3. The config: edit the CIFAR-10 / ResNet-20 one

Everything about the experiment lives in this YAML file: datasets, model, optimizer, schedule and loss. We save
it with `%%writefile` again, this time into the current folder (`/content` on Colab), where step 4 passes it to
the script with `--config`.

In [ ]:
%%writefile lenet5_mnist-hyperparameter_tuning.yaml
dependencies:
  - name: 'lenet5'

datasets:
  mnist_sub_splits: &mnist_sub_splits !import_call
    _name: &dataset_name 'mnist'
    _root: &root_dir !join ['~/datasets/', *dataset_name]
    key: 'torchdistill.datasets.util.split_dataset'
    init:
      kwargs:
        dataset_id: 'original training dataset'
        dataset: !import_call
          key: 'torchvision.datasets.MNIST'
          init:
            kwargs:
              root: *root_dir
              train: True
              download: True
        lengths: [0.9, 0.1]
        generator_seed: 42
        sub_splits_configs:
          - transform: !import_call
              key: 'torchvision.transforms.Compose'
              init:
                kwargs:
                  transforms:
                  - !import_call
                    key: 'torchvision.transforms.RandomCrop'
                    init:
                      kwargs:
                        size: 28
                        padding: 2
                  - !import_call
                    key: 'torchvision.transforms.ToTensor'
                    init:
                  - !import_call
                    key: 'torchvision.transforms.Normalize'
                    init:
                      kwargs: &normalize_kwargs
                        mean: [0.1307]
                        std: [0.3081]
          - transform: !import_call
              key: 'torchvision.transforms.Compose'
              init:
                kwargs:
                  transforms:
                    - !import_call
                      key: 'torchvision.transforms.ToTensor'
                      init:
                    - !import_call
                      key: 'torchvision.transforms.Normalize'
                      init:
                        kwargs: *normalize_kwargs
  &mnist_train mnist/sub_train: !access_by_index_or_key
    data: *mnist_sub_splits
    index_or_key: 0
  &mnist_val mnist/sub_val: !access_by_index_or_key
    data: *mnist_sub_splits
    index_or_key: 1
  &mnist_test mnist/test: !import_call
    key: 'torchvision.datasets.MNIST'
    init:
      kwargs:
        root: *root_dir
        train: False
        download: True
        transform: !import_call
          key: 'torchvision.transforms.Compose'
          init:
            kwargs:
              transforms:
                - !import_call
                  key: 'torchvision.transforms.ToTensor'
                  init:
                - !import_call
                  key: 'torchvision.transforms.Normalize'
                  init:
                    kwargs: *normalize_kwargs

models:
  model:
    key: &student_model_key 'LeNet5'
    kwargs:
      num_classes: 10
    _experiment: &student_experiment !join [*dataset_name, '-', *student_model_key]
    src_ckpt:
    dst_ckpt: !join ['./resource/ckpt/', *dataset_name, '/ce/', *student_experiment, '-hyperparameter_tuning.pt']

train:
  log_freq: 100
  num_epochs: 20
  train_data_loader:
    dataset_id: *mnist_train
    sampler:
      class_or_func: !import_get
        key: 'torch.utils.data.RandomSampler'
      kwargs:
    kwargs:
      batch_size: 128
      num_workers: 16
      pin_memory: True
      drop_last: False
  val_data_loader:
    dataset_id: *mnist_val
    sampler: &val_sampler
      class_or_func: !import_get
        key: 'torch.utils.data.SequentialSampler'
      kwargs:
    kwargs:
      batch_size: 128
      num_workers: 16
      pin_memory: True
      drop_last: False
  model:
    forward_proc: 'forward_batch_only'
    adaptations:
    sequential: []
    wrapper: 'DistributedDataParallel'
    requires_grad: True
    frozen_modules: []
  optimizer:
    key: 'SGD'
    kwargs:
      lr: 0.01
      momentum: 0.9
      weight_decay: 0.0001
  scheduler:
    key: 'MultiStepLR'
    kwargs:
      milestones: [10, 15]
      gamma: 0.1
  criterion:
    key: 'WeightedSumLoss'
    kwargs:
      sub_terms:
        ce:
          criterion:
            key: 'CrossEntropyLoss'
            kwargs:
              reduction: 'mean'
          criterion_wrapper:
            key: 'SimpleLossWrapper'
            kwargs:
              input:
                is_from_teacher: False
                module_path: '.'
                io: 'output'
              target:
                uses_label: True
          weight: 1.0

test:
  test_data_loader:
    dataset_id: *mnist_test
    sampler: *val_sampler
    kwargs:
      batch_size: 1
      num_workers: 16
      pin_memory: True
      drop_last: False

Compare it with the config we started from. The changes are:

- `dependencies` imports `lenet5.py` (step 2),
- `key: 'LeNet5'` picks our registered model,
- the datasets: `torchvision.datasets.MNIST`, 28×28 crops without horizontal flips (flipped digits are not digits),
  and MNIST's mean/std for normalization,
- a shorter schedule: 20 epochs, with the learning rate decayed at epochs 10 and 15,
- the rest just renames `cifar10` to `mnist` in names and dataset ids.

The optimizer, the loss and the training loop are the same as for ResNet-20.

In [ ]:
base_config = 'torchdistill/configs/sample/cifar10/ce/resnet20-hyperparameter_tuning.yaml'
our_config = 'lenet5_mnist-hyperparameter_tuning.yaml'
diff = difflib.unified_diff(Path(base_config).read_text().splitlines(), Path(our_config).read_text().splitlines(),
                            base_config, our_config, n=1, lineterm='')
print('\n'.join(diff))

### A config about train / validation / test

This is the `-hyperparameter_tuning` variant: MNIST's 60,000 training images are split into 54,000 for training and
6,000 for validation. The checkpoint with the best **validation** accuracy is kept, and the 10,000 **test** images are
used only once, at the end. If you try other hyperparameters, compare them on validation accuracy, not test accuracy.

### One Colab-specific change

The config asks for 16 data-loading workers, but a Colab CPU runtime has only 2 cores. We change that number in the
config file. As before, the code stays the same.

In [ ]:
config_text = Path(our_config).read_text()
config_text, num_changes = re.subn(r'num_workers: \d+', 'num_workers: 2', config_text)
Path(our_config).write_text(config_text)
print(f'num_workers set to 2 in {num_changes} data loaders')

## 4. Train with the unchanged `image_classification.py`

The script reads the config, downloads MNIST to `~/datasets/mnist` the first time, trains for 20 epochs while
validating after each one, and saves the best-on-validation checkpoint under `resource/ckpt/mnist/ce/`. At the end it
evaluates that checkpoint on the test set (`Acc@1` on the last line). Expect test accuracy around 99%.

On a CPU runtime you will see a `'pin_memory' argument is set as true but no accelerator is found` warning. It is
harmless: `pin_memory` only speeds up copies to a GPU.

`run_name` names the log file (`log/baseline.log`). In step 6 you give every experiment its own name, so its
log is kept and can be compared with the others.

In [ ]:
run_name = 'baseline'
!python torchdistill/examples/torchvision/image_classification.py --config {our_config} --device {DEVICE} --run_log log/{run_name}.log --seed 42

## 5. Look at the trained model

Training ran in a separate process, so we import the model files listed under `dependencies` here too, so that
their classes (`LeNet5`) are registered in this notebook. Then we rebuild the model from the config, load the saved
checkpoint and label a few test digits. This cell always shows the model of the **last** run of step 4.

In [ ]:
config = load_yaml_file(our_config)
for dependency in config['dependencies']:
    model_file = Path('torchdistill/examples/torchvision', f"{dependency['name']}.py")
    spec = importlib.util.spec_from_file_location(model_file.stem, model_file)
    spec.loader.exec_module(importlib.util.module_from_spec(spec))

model_config = config['models']['model']
model = get_model(model_config['key'], **model_config['kwargs'])
load_ckpt(model_config['dst_ckpt'], model=model, strict=True)
model = model.to(DEVICE).eval()

test_dataset = config['datasets']['mnist/test']
num_shown = 16
images, labels = zip(*[test_dataset[index] for index in range(num_shown)])
with torch.no_grad():
    predictions = model(torch.stack(images).to(DEVICE)).argmax(dim=1).cpu()

fig, axes = plt.subplots(2, num_shown // 2, figsize=(12, 3.5))
for ax, image, label, prediction in zip(axes.flat, images, labels, predictions):
    ax.imshow(image[0], cmap='gray')
    ax.set_title(f'pred {prediction.item()} / true {label}', fontsize=9,
                 color='black' if prediction.item() == label else 'red')
    ax.axis('off')
plt.tight_layout()
plt.show()

## 6. Try it yourself

Now it is your turn to run experiments. Every idea below changes **only the config (step 3) or a model file
(step 2)**, never `image_classification.py`.

### How to run an experiment

1. Edit the config cell in step 3 (or the model cell in step 2) and run it.
2. Run the `num_workers` cell after step 3 again: rerunning step 3 rewrote the file with 16 workers.
3. In step 4, change `run_name` to describe the experiment (for example `'adam'`) and run the cell.
4. Run the **Compare your runs** cell at the end of this section to see all runs side by side.

Some ground rules:

- **Change one thing at a time.** If you change two things and the accuracy moves, you do not know which change did it.
- **Compare runs on validation accuracy.** Test accuracy is the final exam: pick your best setting on validation
  accuracy first, then look at its test accuracy once.
- **Write down what you changed** before you start the next experiment, so you can undo it.
- A 20-epoch run takes a few minutes on a CPU. To explore faster, lower `num_epochs`, but only compare runs that used
  the same number of epochs.

### Warm-up: how much is luck?

Before trying to improve anything, measure the noise. Run the unchanged baseline two more times with a different seed:
change `--seed 42` to `--seed 1`, then `--seed 2`, in step 4 (with `run_name` `'seed1'` and `'seed2'`).

The seed changes the random weight initialization and the order of the training images. The spread of best validation
accuracy across these three runs is your **noise level**. A later change that moves validation accuracy by less than
that has not shown that it helps.

### A. Training hyperparameters (the `train` section of the config)

- **Epochs.** Try `num_epochs: 5` and `num_epochs: 40`. For 40 epochs, also move the learning-rate drops to the same
  relative positions (`milestones: [20, 30]`). After how many epochs does validation accuracy stop improving?
- **Learning rate.** Try `lr: 0.1` and `lr: 0.001` (10× larger and 10× smaller). Watch `loss` in the log of the first
  epoch. What goes wrong at each end?
- **Batch size.** In `train_data_loader`, try `batch_size: 32` and `batch_size: 512`. A larger batch means fewer
  weight updates per epoch. Does a larger batch need a larger learning rate (try `512` with `lr: 0.04`)?
- **Another optimizer.** Replace the `optimizer` block with Adam, which adapts the step size for each weight:

  ```yaml
    optimizer:
      key: 'Adam'
      kwargs:
        lr: 0.001
  ```
- **Another learning-rate schedule.** Replace the `scheduler` block with a cosine schedule, which lowers the learning
  rate smoothly instead of in two steps (`T_max` is the number of epochs):

  ```yaml
    scheduler:
      key: 'CosineAnnealingLR'
      kwargs:
        T_max: 20
  ```

### B. Data augmentation (the `datasets` section of the config)

Augmentation creates slightly different versions of each training image on the fly. Only the **first** transform
list (training) is augmented; validation and test images are left as they are.

- **No augmentation.** Delete the six lines of the `RandomCrop` entry (from `- !import_call` to `padding: 2`). Does the
  training loss go lower while validation accuracy does not?
- **Rotation.** Replace `RandomCrop` with a small random rotation:

  ```yaml
                    - !import_call
                      key: 'torchvision.transforms.RandomRotation'
                      init:
                        kwargs:
                          degrees: 15
  ```
- **Horizontal flips.** Flips are standard for CIFAR-10. Add one after `RandomCrop`, but first **predict** whether it
  helps or hurts on digits, and why:

  ```yaml
                    - !import_call
                      key: 'torchvision.transforms.RandomHorizontalFlip'
                      init:
                        kwargs:
                          p: 0.5
  ```
- **A harder dataset with the same model.** Fashion-MNIST has the same format as MNIST (28×28 grayscale, 10 classes),
  but the images are clothes, not digits. Change `'torchvision.datasets.MNIST'` to `'torchvision.datasets.FashionMNIST'`
  in **both** places, and set its normalization to `mean: [0.2860]` and `std: [0.3530]`. How much lower is the
  accuracy? In step 5 the labels are then class numbers: 0 = T-shirt, 1 = trouser, …, 9 = ankle boot.

### C. Loss function (the `criterion` block of the config)

- **Label smoothing.** Instead of asking the model to be 100% sure of the right class, ask for 90% and spread the rest
  over the other classes. Add one line under `reduction: 'mean'`:

  ```yaml
                reduction: 'mean'
                label_smoothing: 0.1
  ```

### D. Models (step 2)

- **The original LeNet-5.** The 1998 network used `Tanh` activations and average pooling. In the step 2 cell, replace
  every `nn.ReLU(inplace=True)` with `nn.Tanh()` and every `nn.MaxPool2d(2)` with `nn.AvgPool2d(2)`.
- **Batch normalization.** Add `nn.BatchNorm2d(6)` right after the first `nn.Conv2d(...)` and `nn.BatchNorm2d(16)`
  right after the second one.
- **A wider LeNet, as a new model.** Practice the whole recipe from the video. Make a new cell that saves a copy of
  the model under a new file name and class name, with more channels:

  1. Copy the step 2 cell. Change its first line to `%%writefile torchdistill/examples/torchvision/lenet5_wide.py`.
  2. Rename the class to `LeNet5Wide`, and change the channels `Conv2d(1, 6, 5, ...)` → `Conv2d(1, 32, 5, ...)`,
     `Conv2d(6, 16, 5)` → `Conv2d(32, 64, 5)` and `16 * 5 * 5` → `64 * 5 * 5`.
  3. In the config, change `- name: 'lenet5'` to `- name: 'lenet5_wide'` and `key: &student_model_key 'LeNet5'` to
     `'LeNet5Wide'`.

  Is the gain worth the extra size? Count the weights of each model at the end of step 5 with
  `sum(p.numel() for p in model.parameters())`.
- **A model without convolutions.** Do the same with this multi-layer perceptron (file `mlp.py`, class `MLP`):

  ```python
  %%writefile torchdistill/examples/torchvision/mlp.py
  from torch import nn
  from torchdistill.models.registry import register_model


  @register_model
  class MLP(nn.Module):
      def __init__(self, num_classes=10):
          super().__init__()
          self.layers = nn.Sequential(
              nn.Flatten(), nn.Linear(28 * 28, 256), nn.ReLU(inplace=True), nn.Linear(256, num_classes),
          )

      def forward(self, x):
          return self.layers(x)
  ```
  How far behind LeNet-5 is it? Do you see why convolutions suit images?

### Compare your runs

This cell reads every log in `log/` and lists, for each run, the best validation accuracy and the test accuracy of
that checkpoint, best validation first. Choose your best setting by the validation column.

In [ ]:
rows = list()
for log_file in sorted(Path('log').glob('*.log')):
    log = log_file.read_text()
    best_val = re.findall(r'Best top-1 accuracy: [\d.]+ -> ([\d.]+)', log)
    # The test result is the Acc@1 after the test pass
    test_part = log.rsplit('Test: Total time', 1)
    test = re.findall(r'Acc@1 ([\d.]+)', test_part[1]) if len(test_part) == 2 else list()
    rows.append((log_file.stem, float(best_val[-1]) if best_val else None, float(test[0]) if test else None))

print(f'{"run":<24} {"best val acc":>12} {"test acc":>9}')
for name, best_val, test in sorted(rows, key=lambda row: -1 if row[1] is None else row[1], reverse=True):
    print(f'{name:<24} {"-" if best_val is None else f"{best_val:.2f}":>12} {"-" if test is None else f"{test:.2f}":>9}')